# 향수 추천 실행 노트북 (창 버전)

`perfume_recommendation.ipynb`의 두 모델을 불러와 **별도 창(tkinter)**에서 클릭으로 추천을 받는 노트북.

**사용 방법:** 모든 셀 실행(Run All) → 새 창이 뜨면 추천 방식을 클릭해 선택. 창을 닫으면 셀 실행이 끝남.

| 선택 | 모델 | 창에서 하는 일 | 추천 근거 |
|---|---|---|---|
| 콘텐츠 기반 | TF-IDF + 코사인 유사도 | 좋아하는 향수 추가 그리고/또는 원하는 향 종류 선택, 성별 여러 개 선택 | 향 노트·어코드 80% + 같은 브랜드 20% / 고른 향 종류와의 일치도 |
| 협업 필터링 | Item-based KNN | 기존 사용자 선택, 또는 향수에 1~10점 평점 입력 | 비슷하게 평가된 향수들의 평점 가중 평균 |

- tkinter는 파이썬 기본 포함이라 추가 설치 없음
- 모델 코드는 `perfume_recommendation.ipynb`에서 그대로 불러오며 이 노트북에서 수정하지 않음
- 전체 평점으로 학습한 추천 예시이므로 성능 수치는 팀 저장소의 `evaluation/evaluate.ipynb` 참고

### 수정 이력
| 날짜 | 위치 | 내용 | 이유 |
|---|---|---|---|
| 2026-10-31 | 3. 검색용 목록 준비 | `GENDER_OPTIONS`에서 "전체" 제거, `SCENT_FAMILIES`(향 종류 16가지 → 어코드), `top_accords`, `score_for_scents()` 추가 | 향 종류 선택과 성별 복수 선택 지원 |
| 2026-10-31 | 4. 추천 창 `SearchPicker` | 목록 높이를 인자(`height`)로 받도록 변경 | 콘텐츠 기반 화면 공간 확보 |
| 2026-10-31 | 3. 검색용 목록 준비 / 4. 추천 창 | `fr_urls`·`atr_urls` 추가, `fill()`에 `urls` 인자, `open_link()` 추가, 결과 표 더블클릭 연결 | 추천 향수 페이지를 더블클릭으로 바로 열기 위해 |
| 2026-10-31 | 4. 추천 창 `PerfumeApp` | 창 크기 확대, `show_content()`에 향 종류 체크박스·성별 체크박스, 결과 표에 '주요 향' 열, `recommend_content()`에서 향수·향 종류 점수 평균 | 향수와 향 종류를 함께 고르고 성별을 여러 개 고르기 위해 |

코드 안의 `# [수정]`, `# [추가]` 주석에 원래 코드, 바뀐 내용, 이유를 적어 둠. 팀 모델 클래스(`perfume_recommendation.ipynb`)는 수정하지 않음.

## 1. 모델 클래스와 데이터 불러오기

In [ ]:
import ast
import json
from pathlib import Path

import pandas as pd

MODEL_NOTEBOOK = Path("perfume_recommendation.ipynb")
DATA_DIR = Path("data")  # csv 경로가 달라질 시 이 부분에서 경로를 수정

# 모델 노트북의 import와 함수/클래스 정의만 실행 (evaluate.ipynb와 같은 방식)
namespace = {"__name__": "__run_recommender__"}
notebook = json.loads(MODEL_NOTEBOOK.read_text(encoding="utf-8"))
for cell in notebook["cells"]:
    if cell["cell_type"] != "code":
        continue
    try:
        tree = ast.parse("".join(cell.get("source", [])))
    except SyntaxError:
        continue
    selected = [node for node in tree.body if isinstance(
        node, (ast.Import, ast.ImportFrom, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))]
    if selected:
        module = ast.fix_missing_locations(ast.Module(body=selected, type_ignores=[]))
        exec(compile(module, str(MODEL_NOTEBOOK), "exec"), namespace)

load_data = namespace["load_data"]
ContentBasedRecommender = namespace["ContentBasedRecommender"]
ItemKNNRecommender = namespace["ItemKNNRecommender"]

data = load_data(DATA_DIR)
perfumes, ratings, atrafshan = data["content"], data["ratings"], data["atrafshan"]
print(f"Fragrantica 향수 {len(perfumes):,}개 / Atrafshan 평점 {len(ratings):,}건 불러옴")

## 2. 두 모델 학습

In [ ]:
# 모델 노트북과 같은 설정값 사용
KNN_PARAMS = dict(k=20, min_common_users=2, shrinkage=10.0, min_neighbors=1)

content_model = ContentBasedRecommender(brand_weight=0.2).fit(perfumes)
knn_model = ItemKNNRecommender(**KNN_PARAMS).fit(ratings, item_metadata=atrafshan)

print("콘텐츠 기반 TF-IDF 행렬:", content_model.tfidf_matrix_.shape)
print("KNN 사용자 × 향수 평점 행렬:", knn_model.ratings_matrix_.shape)

## 3. 검색용 목록 준비

- Fragrantica 이름은 `encre-noire`처럼 하이픈으로 저장되어 있어 검색할 때 공백으로 바꿔 비교
- 검색 결과는 평가 수가 많은 향수부터 표시. KNN은 평점이 많은 향수일수록 이웃을 찾기 쉬움

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import normalize

TOP_N = 10
MAX_SHOWN = 50
NEW_USER_ID = "atr:user:new"
# [수정] 성별 선택지
#   원래: {"전체": None, "여성": ..., "남성": ..., "유니섹스": ...} 중 하나만 고르는 방식
#   변경: "전체"를 없애고 여성·남성·유니섹스를 각각 체크하는 방식 (3개 모두 체크 = 전체)
#   이유: "남성+유니섹스"처럼 여러 성별을 함께 고를 수 있게 하기 위해
GENDER_OPTIONS = {"여성": "women", "남성": "men", "유니섹스": "unisex"}

# [추가] 향 종류 선택지: Atrafshan 향 계열 16가지 → Fragrantica 어코드 토큰
#   이유: 좋아하는 향수 없이도(또는 향수와 함께) 원하는 향 종류로 추천받기 위해
#   - Atrafshan의 기본 향 계열 16가지를 선택지로 쓰고, 콘텐츠 모델이 쓰는 accord_ 토큰으로 바꿔 검색
#   - 오리엔탈·푸제르·시프레는 Fragrantica에 같은 어코드가 없어 대표 어코드 조합으로 근사 (*)
SCENT_FAMILIES = {
    "플로럴": ["floral", "white_floral"],
    "우디": ["woody"],
    "오리엔탈*": ["amber", "warm_spicy"],
    "아로마틱": ["aromatic"],
    "스파이시": ["warm_spicy", "fresh_spicy"],
    "프루티": ["fruity"],
    "머스크": ["musky"],
    "푸제르*": ["lavender", "aromatic", "mossy"],
    "시프레*": ["mossy", "patchouli", "citrus"],
    "바닐라": ["vanilla"],
    "시트러스": ["citrus"],
    "아쿠아틱": ["aquatic", "marine"],
    "레더": ["leather"],
    "스위트": ["sweet"],
    "그린/허벌": ["green", "herbal"],
    "알데하이드": ["aldehydic"],
}
_vocab = content_model.note_vectorizer_.vocabulary_
_missing = [a for accords in SCENT_FAMILIES.values() for a in accords if "accord_" + a not in _vocab]
assert not _missing, f"TF-IDF 사전에 없는 어코드: {_missing}"

# 콘텐츠 기반 검색 목록: 추천 가능한 Fragrantica 향수
fr_catalog = perfumes.loc[perfumes.perfume_id.isin(content_model.item_index_)].copy()
fr_catalog["n"] = pd.to_numeric(fr_catalog.rating_count, errors="coerce").fillna(0)
fr_catalog["search"] = (fr_catalog.brand + " " + fr_catalog.perfume_name).str.replace("-", " ").str.casefold()
fr_catalog["label"] = fr_catalog.brand + " - " + fr_catalog.perfume_name + "  (" + fr_catalog.gender + ")"
# [추가] 추천 결과 표에 보여줄 주요 어코드 3개
#   이유: 고른 향 종류가 추천 결과에 실제로 반영됐는지 화면에서 바로 확인하기 위해
fr_catalog["top_accords"] = fr_catalog.main_accords.map(lambda s: ", ".join(json.loads(s)[:3]) if s else "")
fr_catalog = fr_catalog.set_index("perfume_id", drop=False)

# 협업 필터링 검색 목록: 평점이 1건 이상 있는 Atrafshan 향수
atr_catalog = atrafshan.assign(n=atrafshan.perfume_id.map(ratings.groupby("perfume_id").size()))
atr_catalog = atr_catalog.loc[atr_catalog.n.notna()].copy()
atr_catalog["search"] = (atr_catalog.brand + " " + atr_catalog.perfume_name).str.casefold()
atr_catalog["label"] = (atr_catalog.brand + " - " + atr_catalog.perfume_name
                        + "  (평점 " + atr_catalog.n.astype(int).astype(str) + "건)")
atr_catalog = atr_catalog.set_index("perfume_id", drop=False)

# [추가] 추천 결과를 더블클릭했을 때 열 향수 페이지 주소
#   이유: 추천받은 향수의 상세 정보(노트, 리뷰 등)를 바로 확인할 수 있게 하기 위해
#   - 콘텐츠 기반 결과: Fragrantica 페이지
#   - 협업 필터링 결과: Fragrantica와 연결된(matched_exact) 향수는 Fragrantica 페이지(영어),
#     연결되지 않은 향수는 Atrafshan 페이지(페르시아어)
fr_urls = perfumes.set_index("perfume_id").url
exact_links = data["mapping"].loc[data["mapping"].match_status.eq("matched_exact")]
atr_urls = atrafshan.set_index("perfume_id").url.copy()
atr_urls.update(exact_links.set_index("atrafshan_id").fragrantica_id.map(fr_urls))

# 기존 사용자 목록: 평점이 많은 순서
user_counts = ratings.groupby("user_id").size().sort_values(ascending=False, kind="stable")
user_labels = [f"{user} (평점 {count}건)" for user, count in user_counts.items()]


def search_catalog(catalog, keyword):
    # 공백으로 나눈 단어가 모두 포함된 향수를 평가 수가 많은 순서로 반환
    mask = pd.Series(True, index=catalog.index)
    for word in keyword.casefold().replace("-", " ").split():
        mask &= catalog.search.str.contains(word, regex=False)
    return catalog.loc[mask].sort_values("n", ascending=False, kind="stable").head(MAX_SHOWN)


# [추가] 고른 향 종류와 각 후보 향수의 코사인 유사도
#   이유: 향 종류만으로도 추천할 수 있게 하기 위해. 팀 모델(ContentBasedRecommender)은 수정하지 않고
#         모델이 학습한 TF-IDF(note_vectorizer_)와 향수 벡터(tfidf_matrix_)를 그대로 사용
#   방법: 고른 향 종류를 "accord_woody accord_citrus ..." 문장으로 만들어 같은 TF-IDF로 변환하고,
#         향수 벡터와 같은 형태(향 블록 √0.8 + 브랜드 블록 0)로 맞춘 뒤 코사인 유사도 계산
def score_for_scents(families, candidate_ids):
    tokens = " ".join("accord_" + a for family in families for a in SCENT_FAMILIES[family])
    query = content_model.note_vectorizer_.transform([tokens]) * np.sqrt(1 - content_model.brand_weight)
    padding = content_model.tfidf_matrix_.shape[1] - query.shape[1]  # 브랜드 블록 자리는 0으로 채움
    query = normalize(hstack([query, csr_matrix((1, padding))], format="csr"))
    rows = content_model.item_index_.get_indexer(candidate_ids)
    scores = (content_model.tfidf_matrix_[rows] @ query.T).toarray().ravel()
    return pd.Series(np.clip(scores, 0, 1), index=pd.Index(candidate_ids, name="perfume_id"), name="score")

## 4. 추천 창 구성

| 화면 | 구성 |
|---|---|
| 시작 | 추천 방식 선택 버튼 2개 |
| 콘텐츠 기반 | 향수 검색 → 더블클릭 또는 [추가] 그리고/또는 향 종류 체크 → 성별 체크(여러 개) → [추천 받기] |
| 협업 필터링 | [기존 사용자] 목록에서 선택, 또는 [직접 평가] 향수 검색 → 슬라이더로 평점 → [평점 추가] → [추천 받기] |

직접 평가 시 입력한 평점을 새 사용자로 학습 데이터에 추가하고 KNN을 다시 학습해서 추천.

In [ ]:
import tkinter as tk
import webbrowser  # [추가] 추천 결과 더블클릭 시 향수 페이지를 기본 브라우저로 열기 위해
from tkinter import messagebox, ttk

try:  # 고해상도 화면에서 글자가 흐려지지 않도록 설정 (Windows)
    import ctypes
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass

FONT = "맑은 고딕"


class SearchPicker(ttk.Frame):
    # 검색창 + 결과 목록. 입력할 때마다 목록을 갱신하고 더블클릭하면 on_pick 호출
    # [수정] 목록 높이를 인자로 받도록 변경
    #   원래: Listbox height=9 고정
    #   변경: height 인자 추가 (기본값 9, 콘텐츠 기반 화면은 6)
    #   이유: 콘텐츠 기반 화면에 향 종류 선택 칸을 넣을 공간을 만들기 위해
    def __init__(self, parent, catalog, title, on_pick, height=9):
        super().__init__(parent)
        self.catalog, self.on_pick, self.shown = catalog, on_pick, []
        ttk.Label(self, text=title).pack(anchor="w")
        self.query = tk.StringVar()
        entry = ttk.Entry(self, textvariable=self.query)
        entry.pack(fill="x", pady=(2, 4))
        entry.bind("<KeyRelease>", lambda event: self.after_idle(self.refresh))
        box = ttk.Frame(self)
        box.pack(fill="both", expand=True)
        self.listbox = tk.Listbox(box, height=height, selectmode="extended", activestyle="none", font=(FONT, 10))
        scroll = ttk.Scrollbar(box, command=self.listbox.yview)
        self.listbox.configure(yscrollcommand=scroll.set)
        self.listbox.pack(side="left", fill="both", expand=True)
        scroll.pack(side="right", fill="y")
        self.listbox.bind("<Double-Button-1>", lambda event: self.on_pick(self.selected_ids()))
        self.refresh()

    def refresh(self):
        keyword = self.query.get().strip()
        found = search_catalog(self.catalog, keyword) if keyword else self.catalog.sort_values(
            "n", ascending=False, kind="stable").head(MAX_SHOWN)
        self.shown = found.perfume_id.tolist()
        self.listbox.delete(0, "end")
        for label in found.label:
            self.listbox.insert("end", label)

    def selected_ids(self):
        return [self.shown[i] for i in self.listbox.curselection()]


class PerfumeApp(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("향수 추천 시스템")
        # [수정] 창 크기  원래: 1180x700, 최소 980x600 → 변경: 1240x800, 최소 1000x720
        #   이유: 콘텐츠 기반 화면에 향 종류 16개 선택 칸과 결과 표의 '주요 향' 열이 추가되어서
        self.geometry("1240x800")
        self.minsize(1000, 720)
        style = ttk.Style(self)
        style.configure(".", font=(FONT, 10))
        style.configure("Title.TLabel", font=(FONT, 18, "bold"))
        style.configure("Head.TLabel", font=(FONT, 13, "bold"))
        style.configure("Big.TButton", font=(FONT, 13, "bold"), padding=(20, 16))
        style.configure("Accent.TButton", font=(FONT, 11, "bold"), padding=(10, 6))
        style.configure("Treeview", rowheight=26)
        style.configure("Treeview.Heading", font=(FONT, 10, "bold"))
        self.body = ttk.Frame(self, padding=16)
        self.body.pack(fill="both", expand=True)
        self.show_start()
        # 노트북에서 실행해도 창이 앞으로 오도록 잠깐 맨 위에 표시
        self.attributes("-topmost", True)
        self.after(600, lambda: self.attributes("-topmost", False))

    # ---------- 공통 ----------
    def clear(self):
        for child in self.body.winfo_children():
            child.destroy()

    def header(self, text):
        bar = ttk.Frame(self.body)
        bar.pack(fill="x", pady=(0, 10))
        ttk.Button(bar, text="← 추천 방식 다시 선택", command=self.show_start).pack(side="left")
        ttk.Label(bar, text=text, style="Head.TLabel").pack(side="left", padx=14)

    def result_panel(self, parent, columns):
        # 추천 결과 표와 아래 설명 문구
        frame = ttk.Frame(parent)
        frame.pack(side="left", fill="both", expand=True, padx=(16, 0))
        self.result_title = ttk.Label(frame, text="추천 결과", style="Head.TLabel")
        self.result_title.pack(anchor="w")
        tree = ttk.Treeview(frame, columns=[c for c, _, _ in columns], show="headings", height=12)
        for name, width, anchor in columns:
            tree.heading(name, text=name)
            tree.column(name, width=width, anchor=anchor, stretch=name in ("브랜드", "향수"))
        tree.pack(fill="both", expand=True, pady=6)
        # [추가] 결과 행 더블클릭 → 향수 페이지 열기 (이유: 추천 향수의 상세 정보를 바로 확인하기 위해)
        tree.bind("<Double-Button-1>", lambda event, t=tree: self.open_link(t))
        self.result_note = ttk.Label(frame, text="", wraplength=640, foreground="#555555", justify="left")
        self.result_note.pack(anchor="w")
        return tree

    # [수정] fill(): 원래 (tree, rows)만 받아 표에 행만 넣음
    #   변경: urls 인자를 추가해 행마다 향수 페이지 주소를 함께 저장 (표에는 보이지 않음)
    #   이유: 더블클릭한 행의 향수 페이지를 열기 위해
    def fill(self, tree, rows, urls=None):
        tree.delete(*tree.get_children())
        self.row_urls = getattr(self, "row_urls", {})
        self.row_urls[str(tree)] = {}
        for row, url in zip(rows, urls if urls is not None else [""] * len(rows)):
            self.row_urls[str(tree)][tree.insert("", "end", values=row)] = url

    # [추가] 선택한 결과 행의 향수 페이지를 기본 브라우저로 열기
    def open_link(self, tree):
        url = getattr(self, "row_urls", {}).get(str(tree), {}).get(tree.focus(), "")
        if url:
            webbrowser.open(url)
        elif tree.focus():
            messagebox.showinfo("링크 없음", "이 향수는 페이지 주소가 없습니다.", parent=self)

    @staticmethod
    def refill(listbox, labels):
        listbox.delete(0, "end")
        for label in labels:
            listbox.insert("end", label)

    # ---------- 시작 화면 ----------
    def show_start(self):
        self.clear()
        ttk.Label(self.body, text="향수 추천 시스템", style="Title.TLabel").pack(pady=(40, 6))
        ttk.Label(self.body, text="추천 방식을 선택하세요").pack(pady=(0, 30))
        row = ttk.Frame(self.body)
        row.pack()
        for title, desc, command in [
            ("콘텐츠 기반 필터링", "좋아하는 향수를 고르면\n향 노트·어코드가 비슷한 향수를 추천\n(TF-IDF + 코사인 유사도)", self.show_content),
            ("협업 필터링", "평점 이력을 바탕으로\n비슷하게 평가된 향수를 추천\n(Item-based KNN)", self.show_knn)]:
            card = ttk.Frame(row, padding=12)
            card.pack(side="left", padx=24)
            ttk.Button(card, text=title, style="Big.TButton", width=18, command=command).pack()
            ttk.Label(card, text=desc, justify="center").pack(pady=10)

    # ---------- 콘텐츠 기반 ----------
    def show_content(self):
        self.clear()
        self.header("콘텐츠 기반 필터링 (TF-IDF + 코사인 유사도)")
        main = ttk.Frame(self.body)
        main.pack(fill="both", expand=True)
        left = ttk.Frame(main, width=430)
        left.pack(side="left", fill="y")
        left.pack_propagate(False)

        self.liked = []
        # [수정] 검색 목록 높이 원래 9줄 → 6줄, 좋아하는 향수 목록 원래 5줄 → 3줄
        #   이유: 아래에 향 종류 선택 칸(16개)을 넣을 공간을 만들기 위해
        self.cb_picker = SearchPicker(left, fr_catalog, "① 좋아하는 향수 검색 (예: encre noire)", self.add_liked, height=6)
        self.cb_picker.pack(fill="both", expand=True)
        ttk.Button(left, text="선택한 향수 추가 ↓", command=lambda: self.add_liked(self.cb_picker.selected_ids())).pack(fill="x", pady=4)

        ttk.Label(left, text="② 내가 좋아하는 향수").pack(anchor="w", pady=(6, 0))
        self.liked_box = tk.Listbox(left, height=3, selectmode="extended", font=(FONT, 10))
        self.liked_box.pack(fill="x")
        ttk.Button(left, text="선택 삭제", command=self.remove_liked).pack(anchor="e", pady=2)

        # [추가] ③ 향 종류 선택 (체크박스 16개, 여러 개 선택 가능)
        #   이유: 향수뿐 아니라 원하는 향 종류로도 추천받을 수 있게 하기 위해. ①②와 ③ 중 하나만 골라도 되고 둘 다 골라도 됨
        ttk.Label(left, text="③ 원하는 향 종류 (여러 개 선택 가능, * 는 비슷한 어코드로 근사)").pack(anchor="w")
        scents = ttk.Frame(left)
        scents.pack(anchor="w", pady=2)
        self.scent_vars = {}
        for i, family in enumerate(SCENT_FAMILIES):
            self.scent_vars[family] = tk.BooleanVar(value=False)
            ttk.Checkbutton(scents, text=family, variable=self.scent_vars[family]).grid(
                row=i // 4, column=i % 4, sticky="w", padx=(0, 8), pady=1)

        # [수정] ④ 성별: 원래 라디오버튼(전체/여성/남성/유니섹스 중 하나) → 체크박스(여성/남성/유니섹스 여러 개)
        #   이유: "남성+유니섹스"처럼 여러 성별을 함께 고르기 위해. 기본값은 3개 모두 체크(= 전체)
        ttk.Label(left, text="④ 성별 (여러 개 선택 가능, 모두 선택 = 전체)").pack(anchor="w", pady=(6, 0))
        genders = ttk.Frame(left)
        genders.pack(anchor="w", pady=2)
        self.gender_vars = {}
        for label in GENDER_OPTIONS:
            self.gender_vars[label] = tk.BooleanVar(value=True)
            ttk.Checkbutton(genders, text=label, variable=self.gender_vars[label]).pack(side="left", padx=(0, 12))
        ttk.Button(left, text="추천 받기", style="Accent.TButton", command=self.recommend_content).pack(fill="x", pady=(10, 0))

        # [수정] 결과 표: 원래 (순위, 브랜드, 향수, 성별, 유사도) → '주요 향'(어코드 3개) 열 추가, '유사도'를 '점수'로 변경
        #   이유: 향 종류를 고른 경우 결과에 그 향이 들어 있는지 확인할 수 있게 하고, 점수가 두 유사도의 평균일 수 있어서
        self.cb_tree = self.result_panel(main, [("순위", 50, "center"), ("브랜드", 140, "w"), ("향수", 220, "w"),
                                                ("성별", 70, "center"), ("주요 향", 210, "w"), ("점수", 70, "center")])
        self.result_note.configure(text="점수: 좋아하는 향수로 만든 향 프로필과의 코사인 유사도, 또는 고른 향 종류와의 코사인 유사도 (0~1). "
                                        "향수와 향 종류를 둘 다 고르면 두 유사도의 평균. 향수 유사도는 향 노트·어코드 80%, 같은 브랜드 20% 반영.\n"
                                        "결과 행을 더블클릭하면 Fragrantica 향수 페이지가 브라우저에서 열림.")

    def add_liked(self, ids):
        self.liked += [pid for pid in ids if pid not in self.liked]
        self.refill(self.liked_box, fr_catalog.loc[self.liked, "label"])

    def remove_liked(self):
        drop = {self.liked[i] for i in self.liked_box.curselection()}
        self.liked = [pid for pid in self.liked if pid not in drop]
        self.refill(self.liked_box, fr_catalog.loc[self.liked, "label"])

    # [수정] 콘텐츠 기반 추천 계산
    #   원래: 좋아하는 향수가 필수, 성별 하나로 후보 제한, content_model.recommend_for_likes()로 Top-10
    #   변경: 1) 좋아하는 향수와 향 종류 중 하나 이상이면 추천 (둘 다 가능)
    #         2) 체크한 성별들(여러 개)로 후보 제한
    #         3) 향수 유사도는 팀 모델의 score_for_likes(), 향 종류 유사도는 score_for_scents()로 계산하고,
    #            둘 다 고르면 두 점수를 평균. 정렬 규칙(점수 내림차순, 동점은 ID 순)과 좋아하는 향수 제외는 원래와 같음
    #   이유: 향수와 향 종류를 함께 고르고, 성별을 여러 개 고를 수 있게 하기 위해
    def recommend_content(self):
        families = [family for family, var in self.scent_vars.items() if var.get()]
        genders = [GENDER_OPTIONS[label] for label, var in self.gender_vars.items() if var.get()]
        if not self.liked and not families:
            messagebox.showinfo("선택 필요", "좋아하는 향수나 원하는 향 종류를 하나 이상 골라 주세요.", parent=self)
            return
        if not genders:
            messagebox.showinfo("성별 선택", "성별을 하나 이상 선택해 주세요.", parent=self)
            return
        candidates = fr_catalog.perfume_id[fr_catalog.gender.isin(genders)].tolist()
        parts = []
        if self.liked:
            parts.append(content_model.score_for_likes(self.liked, candidate_ids=candidates))
        if families:
            parts.append(score_for_scents(families, candidates))
        scores = sum(parts) / len(parts)
        scores = scores.loc[~scores.index.isin(self.liked) & scores.gt(1e-8)]
        ranked = (scores.rename_axis("perfume_id").reset_index()
                  .sort_values(["score", "perfume_id"], ascending=[False, True], kind="stable").head(TOP_N))
        info = fr_catalog.loc[ranked.perfume_id]

        basis = []
        if self.liked:
            basis.append(f"좋아하는 향수 {len(self.liked)}개")
        if families:
            basis.append("향 종류 " + ", ".join(families))
        gender_text = "전체" if len(genders) == len(GENDER_OPTIONS) else "+".join(
            label for label, var in self.gender_vars.items() if var.get())
        self.result_title.configure(text=f"추천 결과 ({gender_text}): {' / '.join(basis)} 기준 Top-{len(ranked)}")
        # [수정] urls 인자 추가: 원래 fill(tree, rows) → fill(tree, rows, urls). 이유: 더블클릭으로 Fragrantica 페이지 열기
        self.fill(self.cb_tree, [(i, b, n, g, a, f"{s:.3f}") for i, (b, n, g, a, s) in enumerate(
            zip(info.brand, info.perfume_name, info.gender, info.top_accords, ranked.score), 1)],
            urls=[fr_urls.get(pid, "") for pid in ranked.perfume_id])

    # ---------- 협업 필터링 ----------
    def show_knn(self):
        self.clear()
        self.header("협업 필터링 (Item-based KNN)")
        main = ttk.Frame(self.body)
        main.pack(fill="both", expand=True)
        left = ttk.Frame(main, width=430)
        left.pack(side="left", fill="y")
        left.pack_propagate(False)

        self.knn_mode = tk.StringVar(value="new")
        modes = ttk.Frame(left)
        modes.pack(anchor="w", pady=(0, 6))
        ttk.Radiobutton(modes, text="직접 평가하기", value="new", variable=self.knn_mode,
                        command=self.switch_knn_mode).pack(side="left", padx=(0, 14))
        ttk.Radiobutton(modes, text="기존 Atrafshan 사용자", value="existing", variable=self.knn_mode,
                        command=self.switch_knn_mode).pack(side="left")

        # 직접 평가: 향수 검색 → 평점 슬라이더 → 내 평점 목록
        self.new_frame = ttk.Frame(left)
        self.my_ratings = {}
        self.atr_picker = SearchPicker(self.new_frame, atr_catalog, "① 평가할 향수 검색 (예: aventus)",
                                       lambda ids: self.add_rating())
        self.atr_picker.pack(fill="both", expand=True)
        score_row = ttk.Frame(self.new_frame)
        score_row.pack(fill="x", pady=4)
        ttk.Label(score_row, text="② 평점").pack(side="left")
        self.score = tk.IntVar(value=8)
        tk.Scale(score_row, from_=1, to=10, orient="horizontal", variable=self.score,
                 length=200, font=(FONT, 9)).pack(side="left", padx=6)
        ttk.Button(score_row, text="평점 추가 ↓", command=self.add_rating).pack(side="left")
        ttk.Label(self.new_frame, text="③ 내 평점 (평점이 많은 향수를 여러 개 평가할수록 이웃을 찾기 쉬움)").pack(anchor="w")
        self.rating_box = tk.Listbox(self.new_frame, height=5, selectmode="extended", font=(FONT, 10))
        self.rating_box.pack(fill="x")
        ttk.Button(self.new_frame, text="선택 삭제", command=self.remove_rating).pack(anchor="e", pady=2)

        # 기존 사용자: 사용자 선택 → 평점 이력 표시
        self.old_frame = ttk.Frame(left)
        ttk.Label(self.old_frame, text="① 사용자 선택 (평점이 많은 순서)").pack(anchor="w")
        self.user_choice = ttk.Combobox(self.old_frame, values=user_labels, state="readonly")
        self.user_choice.current(0)
        self.user_choice.pack(fill="x", pady=(2, 6))
        self.user_choice.bind("<<ComboboxSelected>>", lambda event: self.show_history())
        ttk.Label(self.old_frame, text="이 사용자의 평점 이력").pack(anchor="w")
        self.history_box = tk.Listbox(self.old_frame, height=14, font=(FONT, 10))
        self.history_box.pack(fill="both", expand=True)

        self.knn_button = ttk.Button(left, text="추천 받기", style="Accent.TButton", command=self.recommend_knn)
        self.knn_tree = self.result_panel(main, [("순위", 50, "center"), ("브랜드", 170, "w"), ("향수", 260, "w"),
                                                 ("예상 평점", 90, "center"), ("근거 이웃 수", 100, "center")])
        self.result_note.configure(text="예상 평점: 내가 평가한 향수 중 이 향수와 비슷하게 평가된 향수(이웃)의 평점을 유사도로 가중 평균한 값 (1~10). "
                                        "근거 이웃 수가 1이면 내 평점 하나만 보고 예측한 것이라 근거가 약함.\n"
                                        "결과 행을 더블클릭하면 향수 페이지가 열림 (Fragrantica와 연결된 향수는 Fragrantica, 아니면 Atrafshan).")
        self.show_history()
        self.switch_knn_mode()

    def switch_knn_mode(self):
        self.knn_button.pack_forget()
        self.new_frame.pack_forget()
        self.old_frame.pack_forget()
        frame = self.new_frame if self.knn_mode.get() == "new" else self.old_frame
        frame.pack(fill="both", expand=True)
        self.knn_button.pack(fill="x", pady=(10, 0))

    def add_rating(self):
        ids = self.atr_picker.selected_ids()
        if not ids:
            messagebox.showinfo("향수 선택", "목록에서 평가할 향수를 선택해 주세요.", parent=self)
            return
        for pid in ids:
            self.my_ratings[pid] = int(self.score.get())
        self.refresh_ratings()

    def remove_rating(self):
        keys = list(self.my_ratings)
        for i in self.rating_box.curselection():
            self.my_ratings.pop(keys[i], None)
        self.refresh_ratings()

    def refresh_ratings(self):
        self.refill(self.rating_box, [f"{score}점  {atr_catalog.at[pid, 'brand']} - {atr_catalog.at[pid, 'perfume_name']}"
                                      for pid, score in self.my_ratings.items()])

    def selected_user(self):
        return user_counts.index[self.user_choice.current()]

    def show_history(self):
        history = ratings.loc[ratings.user_id.eq(self.selected_user())].merge(
            atrafshan[["perfume_id", "brand", "perfume_name"]], on="perfume_id")
        history = history.sort_values("scent_rating", ascending=False, kind="stable")
        self.refill(self.history_box, [f"{r.scent_rating:g}점  {r.brand} - {r.perfume_name}"
                                       for r in history.itertuples()])

    def recommend_knn(self):
        if self.knn_mode.get() == "existing":
            model, user_id, who = knn_model, self.selected_user(), self.selected_user()
        else:
            if not self.my_ratings:
                messagebox.showinfo("평점 입력", "향수를 하나 이상 평가해 주세요.", parent=self)
                return
            # 입력한 평점을 새 사용자로 추가해 KNN을 다시 학습
            new_rows = pd.DataFrame({"user_id": NEW_USER_ID, "perfume_id": list(self.my_ratings),
                                     "scent_rating": list(self.my_ratings.values())})
            train = pd.concat([ratings[new_rows.columns], new_rows], ignore_index=True)
            model = ItemKNNRecommender(**KNN_PARAMS).fit(train, item_metadata=atrafshan)
            user_id, who = NEW_USER_ID, f"내 평점 {len(self.my_ratings)}건"
        result = model.recommend(user_id, top_n=TOP_N)
        # [수정] urls 인자 추가: 원래 fill(tree, rows) → fill(tree, rows, urls). 이유: 더블클릭으로 향수 페이지 열기
        self.fill(self.knn_tree, [(i, r.brand, r.perfume_name, f"{r.predicted_rating:.2f}", r.n_neighbors)
                                  for i, r in enumerate(result.itertuples(), 1)],
                  urls=[atr_urls.get(pid, "") for pid in result.perfume_id])
        if result.empty:
            self.result_title.configure(text="추천 결과 없음")
            messagebox.showinfo("추천 불가", "이웃을 찾지 못했습니다. 평점 수가 많은 향수를 더 평가해 보세요.", parent=self)
        else:
            self.result_title.configure(text=f"추천 결과: {who} 기준 Top-{len(result)}")

## 5. 실행

아래 셀을 실행하면 추천 창이 뜸. 창을 닫으면 셀 실행이 끝남.

In [ ]:
app = PerfumeApp()
app.mainloop()